<a href="https://colab.research.google.com/github/sarahliu-mma/Quantitative-Trading-Pipeline_FIAM-Hackathon-2026/blob/main/notebooks/02_data_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Engineering


### Build stock panel data
Output:
*   Identifier: primary key(permno, eom)
*   Features: 147 JKP features
*   Target: 'ret_exc_lead1m'
*   Auxiliary columns

### Deciding the splitting period
*   Training：2015-01 ~ 2018-12
*   Validation：2019-01 ~ 2020-12
*   Testing：2021-01 ~ 2026-08

### Sanity Check:
* No duplicated primary key





#### 1. Setting & Load the Data

In [1]:
import numpy as np
import pandas as pd

In [2]:
dt = pd.read_parquet("/content/drive/MyDrive/FIAM Hackathon/chars_final_with_names.parquet")
jkp_features = pd.read_csv("/content/drive/MyDrive/FIAM Hackathon/factor_char_list.csv")

In [3]:
dt.head()

,id,permno,permco,gvkey,iid,excntry,exch_main,common,primary_sec,bidask,...,qmj_prof,qmj_growth,qmj_safety,qmj,me_lag1,ticker,company_name,ticker_name_reference_date,ticker_name_source,ticker_name_status
0,10001,10001,7953,012994,01,USA,1,1,1,0,...,-0.793438,-1.174184,-0.554461,-1.339223,115.57776,EGAS,GAS NATURAL INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
1,10025,10025,7975,011903,01,USA,1,1,1,0,...,-0.659718,-1.648672,-1.076800,-1.539792,295.46015,AEPI,A E P INDUSTRIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
2,10026,10026,7976,012825,01,USA,1,1,1,0,...,1.291849,-1.083642,1.560033,1.060720,2031.60606,JJSF,J & J SNACK FOODS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
3,10028,10028,7978,012096,01,USA,1,1,1,0,...,0.288480,1.542084,-0.634534,0.691674,14.91328,DGSE,D G S E COMPANIES INC,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month
4,10032,10032,7980,012945,01,USA,1,1,1,0,...,-0.891624,-0.555288,0.647570,-0.687090,1384.86205,PLXS,PLEXUS CORP,2015-01-30,CRSP_monthly_security_history,verified_in_observation_month


2. Build panel data

In [4]:
panel_dt = pd.DataFrame()
panel_dt = dt[['permno','eom']]
panel_dt[jkp_features['variable'].tolist()] = dt[jkp_features['variable'].tolist()]

/tmp/ipykernel_4663/766622789.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_dt[jkp_features['variable'].tolist()] = dt[jkp_features['variable'].tolist()]
/tmp/ipykernel_4663/766622789.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_dt[jkp_features['variable'].tolist()] = dt[jkp_features['variable'].tolist()]
/tmp/ipykernel_4663/766622789.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value inst

In [5]:
panel_dt.head()

,permno,eom,age,aliq_at,aliq_mat,ami_126d,at_be,at_gr1,at_me,at_turnover,...,taccruals_at,taccruals_ni,tangibility,tax_gr1a,turnover_126d,turnover_var_126d,z_score,zero_trades_126d,zero_trades_21d,zero_trades_252d
0,10001,2015-01-31,349,0.527883,0.545502,0.024417,1.883066,0.059447,1.942361,0.683234,...,-0.058428,-2.179343,0.547004,-0.005906,0.004503,0.977763,1.777709,0.006047,0.006052,0.004815
1,10025,2015-01-31,361,0.577454,0.448423,0.022546,5.879399,-0.052203,1.754731,2.597666,...,-0.096701,-7.849619,1.010645,-0.020470,0.004044,0.819247,4.027290,0.006488,0.005533,0.005562
2,10026,2015-01-31,361,0.571576,0.216947,0.001615,1.156135,0.066340,0.383311,1.363699,...,-0.041957,-0.417445,0.707691,0.002787,0.002971,0.698086,12.093942,0.007420,0.006055,0.007454
3,10028,2015-01-31,361,0.522485,0.329818,9.495816,3.174725,-0.267115,0.985359,3.414208,...,-0.032329,-0.113055,0.576669,-0.017429,0.001348,1.302835,3.103010,1.342219,2.109600,1.096408
4,10032,2015-01-31,349,0.773526,0.548978,0.001747,1.959831,0.060762,1.229565,1.652661,...,0.059580,1.005301,0.762701,0.001981,0.005154,0.452238,4.287508,0.005454,0.004392,0.005735


In [6]:
panel_dt[['ret_exc_lead1m']] = dt[['ret_exc_lead1m']]

/tmp/ipykernel_4663/464045425.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  panel_dt[['ret_exc_lead1m']] = dt[['ret_exc_lead1m']]
/tmp/ipykernel_4663/464045425.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  panel_dt[['ret_exc_lead1m']] = dt[['ret_exc_lead1m']]


3. Sanity Check

In [7]:
panel_dt.duplicated(subset=['permno','eom']).sum()

np.int64(0)